In [ ]:
    # Insert NASA-ASFCatalogue Token 
import os
os.environ['EARTHDATA_TOKEN'] = 'Insert Key'  

In [ ]:
     # Load Modules

import odc.geo  # noqa
from odc.algo import mask_cleanup
from odc.stac import configure_rio, load
from pystac_client import Client
import datetime
import os
import rioxarray as rxr
import matplotlib.pyplot as plt
import xarray as xr
import time
import requests
from requests.exceptions import HTTPError
from tqdm import tqdm
import logging
import geopandas as gpd
import csv
print(xr.__version__)

In [ ]:
    # Search collections

catalog = "https://cmr.earthdata.nasa.gov/cloudstac/LPCLOUD/"

# Searching across both landsat and sentinel at 30 m
collections = ["HLSS30.v2.0", "HLSL30.v2.0"]

client = Client.open(catalog)
collections

In [ ]:
    # Add Search data [ Year, Name ROI, Dates]

# Specify MR year
year = '2026'

# Specify name of Region of Interest 

roi = 'Kuamut'

# Select dates

# Construct date string for datetime range
start_date = "2026-01-01"
end_date = '2026-03-02'#"2025-01-28"
#end_date = datetime.date.today().strftime("%Y-%m-%d")

date_string = f"{start_date}/{end_date}"



In [ ]:
# Select ROI
    # Kuamut
ll = (4.932133, 117.185748)
ur = (5.311166, 117.632919)

bbox = [ll[1], ll[0], ur[1], ur[0]]


# Search for items in the collection
items = client.search(
    collections=collections, 
    bbox=bbox, 
    datetime=date_string).items()


In [ ]:
    # Print search
    
items = [i for i in items]
print(f"Found {len(items)} items")
print(items)
for i in items:
  print(i.get_collection())

In [ ]:
    # read bands (Sentinel)
items[0].assets

In [ ]:
    # read bands (Landsat)
items[1].assets

In [ ]:
# Filtering bands

s30_bands = ['B12', 'B11', 'B8A', 'B04', 'B03','B02', 'B01', 'Fmask']    # S30 bands for EVI calculation and quality filtering -> SWIR2, SWIR 1, NIR, RED, GREEN, BLUE, Coastal, Quality 
l30_bands = ['B07', 'B06', 'B05', 'B04', 'B03','B02', 'B01', 'Fmask']    # L30 bands for EVI calculation and quality filtering -> SWIR2, SWIR 1, NIR, RED, GREEN, BLUE, Coastal, Quality

l30_bands #(example)

In [ ]:
# And now to loop through and filter the items collection by bands:
new_band_links = []

for i in items:
    if i.collection_id == 'HLSS30_2.0':
        new_bands = s30_bands
    elif i.collection_id == 'HLSL30_2.0':
        new_bands = l30_bands
    else:
        print(f"Skipping unknown collection: {i.collection_id}")
        continue  # Ensures 'new_bands' is always defined before use

    for a in i.assets:
        if any(b == a for b in new_bands):
            new_band_links.append(i.assets[a].href)

In [ ]:
items[1].assets

In [ ]:
# Rename all bands

for i in items:
    if i.collection_id == 'HLSS30_2.0':
        # Renaming bands for HLSS30.v2.0 items
        band_mapping = {'B12': 'SWIR2', 'B11': 'SWIR 1', 'B8A': 'NIR', 'B04': 'RED', 'B03': 'GREEN', 'B02': 'BLUE', 'B01': 'Coastal', 'Fmask': 'Fmask'}
    elif i.collection_id == 'HLSL30_2.0':
        # Renaming bands for HLSL30.v2.0 items
        band_mapping = {'B07': 'SWIR2', 'B06': 'SWIR 1', 'B05': 'NIR', 'B04': 'RED', 'B03': 'GREEN', 'B02': 'BLUE', 'B01': 'Coastal', 'Fmask': 'Fmask'}

    # Create a copy of the keys to avoid RuntimeError
    asset_keys = list(i.assets.keys())
    for a in asset_keys:
        if a in band_mapping:
            # Update the band name
            i.assets[band_mapping[a]] = i.assets.pop(a)

In [ ]:
items[2].assets

In [ ]:
# Configure GDAL. You need to export your earthdata token as an environment variable.
header_string = f"Authorization: Bearer {os.environ['EARTHDATA_TOKEN']}"
configure_rio(cloud_defaults=True, GDAL_HTTP_HEADERS=header_string)

data = load(
    items,
    bbox=bbox,
    crs="epsg:32650",
    resolution=30,
    chunks={"x": 2500, "y": 2500, "time": 1},
    groupby="solar_day",
    bands=["SWIR2", "SWIR 1", "NIR", "RED", "GREEN", "BLUE", "Coastal", "Fmask"],
)

# Get cloud  mask bitfields
# I think 1 is cloud, but I can't find docs...
# And bit 2 is a mess, but might be cloud shadow... not using that
# I can't actually find the cloud shadow bit
mask_bitfields = [0, 1, 3]
bitmask = 0
for field in mask_bitfields:
    bitmask |= 1 << field

# Get cloud mask
cloud_mask = data["Fmask"].astype(int) & bitmask != 0

# Contract and then expand the cloud mask to remove small areas
dilated = mask_cleanup(cloud_mask, [("opening", 2), ("dilation", 3)])

masked = data.where(~dilated)
masked

In [ ]:
import geopandas as gpd
import csv
import rioxarray
import xarray as xr
import numpy as np

# Load the shapefile
shapefile_path = r"C:\Users\JavierRuizRamos\OneDrive - Permian Global Research Limited\Desktop\Kuamut\MR1_Monitoring Report 2022\Deforestation Kuamut\Kuamut GIS\KuamutProjectArea\Kuamut_ProjectArea.shp"
gdf = gpd.read_file(shapefile_path)

# Define a list to store the data
csv_data = []

# Load your masked data (replace with actual data loading logic)
# masked = xr.open_dataset('path_to_your_masked_data.nc')

# Assuming masked is an xarray Dataset with a time dimension
for i, (time, image) in enumerate(masked.groupby("time")):
    # Clip the image to the extent of the project area
    clipped_image = image.rio.clip(gdf.geometry)

    # Calculate the total number of pixels in the clipped image
    total_pixels = clipped_image["BLUE"].size

    # Create a mask for non-NaN (non-missing) pixels
    non_nan_mask = clipped_image["BLUE"].notnull()

    # Count the number of non-NaN pixels
    non_nan_pixels = non_nan_mask.sum().compute()  # Compute Dask array to get scalar value

    # Calculate the percentage of area covered by non-NaN pixels
    percentage_covered = (non_nan_pixels / total_pixels) * 100

    # Convert numpy.datetime64 to datetime object
    time_dt = np.datetime_as_string(time, unit='s')

    # Format time to match the desired format (e.g., '2024-06-28T02:46:26.218000000')
    formatted_time = time_dt

    # Append the data to the list
    csv_data.append([f"Image {i+1}", formatted_time, f"{percentage_covered:.2f}%"])

# Define the filename for the CSV file

csv_filename = f"HSL_CompositeMaskedImageryused{roi}{year}.csv"

# Write the data to the CSV file
with open(csv_filename, 'w', newline='') as csvfile:
    # Create a CSV writer object
    csv_writer = csv.writer(csvfile)

    # Write the header
    csv_writer.writerow(["Image", "Date", "Percentage of PA with imagery used"])

    # Write the data
    csv_writer.writerows(csv_data)

print(f"CSV file '{csv_filename}_CloudTest' has been created successfully.")

import warnings
warnings.filterwarnings("ignore", message="invalid value encountered in cast")



In [ ]:
    # Show image test of masked images
    
masked[["RED", "GREEN", "BLUE"]].isel(time=slice(-12, None)).to_array().plot.imshow(
    col="time", col_wrap=4, vmin=0, vmax=3000
)

In [ ]:
# Enable logging to track progress
logging.basicConfig(format='%(asctime)s - %(message)s', level=logging.INFO)

def retry_with_backoff(url, max_retries=5):
    """Retry a request with exponential backoff."""
    for i in range(max_retries):
        try:
            response = requests.get(url)
            response.raise_for_status()
            return response
        except HTTPError as e:
            logging.warning(f"HTTP error ({e.response.status_code}) - {url}. Retrying again in {2**i} secs")
            time.sleep(2**i)
    logging.error(f"Failed to retrieve data from {url} after {max_retries} retries")
    return None

# Create a simple cloud-free median now we have masked data
logging.info("Computing cloud-free median...")
num_steps = masked.time.size

# Initialize progress bar
with tqdm(total=num_steps) as pbar:
    median_list = []
    for step in range(num_steps):
        try:
            # Select the time step and add to the list
            median_list.append(masked.isel(time=step))
            # Update progress bar
            pbar.update(1)
        except Exception as e:
            logging.error(f"Error processing step {step}: {e}")

# Concatenate the individual time steps and compute the median along the time dimension
median = xr.concat(median_list, dim='time').median('time').compute()

logging.info("Cloud-free median computation completed.")


In [ ]:
    # Create a simple cloud-free median now we have masked data (alternative)
median = masked.median("time").compute()


In [ ]:
    # Plot the median. This is just one month, so we expect
# some areas to be missing due to clouds
rgb = median[["RED", "GREEN", "BLUE"]].to_array()
rgb.plot.imshow(size=10, vmin=0, vmax=1000)

In [ ]:
    # Show Image and Total ROI area covered by image composite 

roi = 'Kuamut'

# Load the shapefile
shapefile_path = r"C:\Users\JavierRuizRamos\OneDrive - Permian Global Research Limited\Desktop\Kuamut\MR1_Monitoring Report 2022\Deforestation Kuamut\Kuamut GIS\KuamutProjectArea\Kuamut_ProjectArea.shp"
gdf = gpd.read_file(shapefile_path)

# Calculate the area of the 'Kuamut project area' polygon
project_area_sqm = gdf.geometry.to_crs('EPSG:32650').area.sum()

# Calculate the area covered by the satellite image within the 'Kuamut project area'
clipped_image = median.rio.clip(gdf.geometry)
resolution = 30  # Example resolution, replace with actual resolution if known
image_area_sqm = (clipped_image["RED"].count() * resolution ** 2)

# Calculate the percentage of the area covered by the satellite image relative to the total area of the 'Kuamut project area'
percentage_covered = (image_area_sqm / project_area_sqm) * 100

# Plot the satellite image
fig, ax = plt.subplots(figsize=(10, 10))
vmin, vmax = median["RED"].min(), median["RED"].max()
rgb = median[["RED", "GREEN", "BLUE"]].to_array()
rgb.plot.imshow(ax=ax, vmin=0, vmax=1000, extent=(median.x.min(), median.x.max(), median.y.min(), median.y.max()))
gdf.plot(ax=ax, color='none', edgecolor='red')

# Annotate the plot with the calculated percentage of coverage
plt.text(0.95, 0.95, f'Total Coverage: {percentage_covered:.2f}%', horizontalalignment='right', verticalalignment='top', transform=ax.transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

# Set the plot title
plt.title(f'HSL Cloud-Free Composite {roi} {year}_Full')

# Rename the axis labels
plt.xlabel('X coordinates')
plt.ylabel('Y coordinates')

# Save the plot as a high-resolution JPEG image
output_path = f"HSL_CloudFreeComposite_{roi}{year}_Full.jpg"
plt.savefig(output_path, dpi=300)  # Set the dpi parameter to adjust the resolution

plt.show()




In [ ]:
    # Normalised Difference Vegetation Index (NDVI)

# Select bands B05 (NIR) and B04 (Red) from the median image
nir_median = median['NIR']
red_median = median['RED']

# Calculate NDVI for the median image
ndvi_median = (nir_median - red_median) / (nir_median + red_median)

# Add NDVI as a new variable to the median image dataset
median_with_ndvi = median.assign(NDVI=ndvi_median)

# Set up a larger figure size
plt.figure(figsize=(10, 8))

# Visualize the NDVI band with scale adjusted to 0 to 1
plt.imshow(median_with_ndvi.NDVI, cmap='RdYlGn', extent=(0, median_with_ndvi.NDVI.shape[1], 0, median_with_ndvi.NDVI.shape[0]), vmin=0, vmax=1)
plt.colorbar(label='NDVI')
plt.title(f'NDVI of Median Composite {roi} {year}_Full')
plt.show()

In [ ]:
    # Enhanced Vegetation Index (EVI)
    
# Constants for EVI calculation
G = 2.5
C1 = 6
C2 = 7.5
L = 1

import matplotlib.pyplot as plt

# Select bands B05 (NIR) and B04 (Red) from the median image
nir = median_with_ndvi['NIR']
red = median_with_ndvi['RED']
blue = median_with_ndvi['BLUE']

    # Calculate EVI
evi_median = G * ((nir - red) / (nir + C1 * red - C2 * blue + L))

# Add NDVI as a new variable to the median image dataset
median_with_evi = median_with_ndvi.assign(EVI=evi_median)

# Set up a larger figure size
plt.figure(figsize=(10, 8))

# Visualize the NDVI band with scale adjusted to 0 to 1
plt.imshow(median_with_evi.EVI, cmap='RdYlGn', extent=(0, median_with_evi.EVI.shape[1], 0, median_with_evi.EVI.shape[0]), vmin=0, vmax=3)
plt.colorbar(label='EVI')
plt.title(f'EVI of Median Composite {roi} {year}_Full')
plt.show()
  
 


In [ ]:
    # Save Composite image with both NDVI and EVI indexes

import os
import rioxarray as rxr
# Assuming median_with_ndvi contains the raster dataset

# Get the current working directory
current_dir = os.getcwd()

# Define the file path for the COG
cog_path = os.path.join(current_dir, f"{roi}_HSL_Median{year}_Full.tif")

# Save the dataset as a Cloud Optimized GeoTIFF
median_with_evi.rio.to_raster(cog_path, driver="GTiff", tif_cog_profile="deflate")